# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access the human-readable JSON metadata
metadata_json = dataset.metadata.to_json(indent=2)
metadata = json.loads(metadata_json)

print(f"Dataset title: {metadata.get('name', '')}\nDescription: {metadata.get('description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll use the `@id` of each record set, field, and column for referencing. This ensures we address entities uniquely according to the Croissant specification.

In [ ]:
# Extract top-level record sets (cr:RecordSet objects)
from pprint import pprint

schema_dict = json.loads(dataset.metadata.raw)

def get_entities_by_type(schema, entity_type):
    """Recursively find entities with a given @type."""
    entities = []
    if isinstance(schema, dict):
        if '@type' in schema and (
            (isinstance(schema['@type'], list) and entity_type in schema['@type']) or
            (entity_type == schema['@type'])
        ):
            entities.append(schema)
        for v in schema.values():
            entities += get_entities_by_type(v, entity_type)
    elif isinstance(schema, list):
        for s in schema:
            entities += get_entities_by_type(s, entity_type)
    return entities

# Record sets are of type 'cr:RecordSet'
record_sets = get_entities_by_type(schema_dict, 'cr:RecordSet')
if not record_sets:
    print('No RecordSets found in the schema.')
else:
    print(f"Found {len(record_sets)} RecordSet(s):\n")
    for rec in record_sets:
        print(f"@id: {rec.get('@id')}")
        print(f"  Name: {rec.get('name')}\n  Description: {rec.get('description','')}")
        # If present, print fields/columns
        fields = rec.get('field') or rec.get('fields') # Some schemas use 'field', some 'fields'
        if fields:
            if not isinstance(fields, list):
                fields = [fields]
            print(f"  Fields (@id): {[field.get('@id') if isinstance(field, dict) else field for field in fields]}")
        columns = rec.get('column') or rec.get('columns')
        if columns:
            if not isinstance(columns, list):
                columns = [columns]
            print(f"  Columns (@id): {[column.get('@id') if isinstance(column, dict) else column for column in columns]}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

<details>
<summary>🛈 If you don't see record set IDs above, refer to the Croissant JSON-LD file for specific `@id` values.</summary>
</details>

In [ ]:
# >>> Example workflow for loading all RecordSets into DataFrames <<<
# (Replace the following with your record_set(s) of interest using their @id)

# If there are no explicit RecordSets, try the main Dataset-level records
from collections import OrderedDict

dataframes = {}

# If no record sets are found (many bio datasets use just the top-level or distribution),
# try to get all available record sets from the metadata, otherwise use distribution directly
if record_sets:
    record_set_ids = [rec["@id"] for rec in record_sets]
else:
    record_set_ids = []
    # fallback: attempt to use distribution files as default record set
    print("No explicit RecordSets defined. Trying to extract from distribution(s)...")
    dists = schema_dict.get('distribution', [])
    for d in dists:
        if isinstance(d, dict) and '@id' in d:
            record_set_ids.append(d['@id'])

print('RecordSet IDs (or Distribution IDs) considered for record extraction:')
for rsid in record_set_ids:
    print(f"  {rsid}")

for record_set in record_set_ids:
    try:
        # This will try to load all records for the record_set by @id
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        if not df.empty:
            dataframes[record_set] = df
            print(f"Loaded {len(df)} records from record set @id '{record_set}'")
        else:
            print(f"No data found for record set {record_set}")
    except Exception as e:
        print(f"Failed loading record set {record_set}: {e}")

if dataframes:
    example_id = list(dataframes.keys())[0]
    print(f"Columns in DataFrame for record set @id '{example_id}': \n{dataframes[example_id].columns.tolist()}")
    dataframes[example_id].head()
else:
    print("No dataframes loaded! Check record set IDs and data availability.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Use only `@id` columns for reference (these are the DataFrame columns corresponding to field or column `@id`s in the schema).

In [ ]:
# EDA Example: demonstrate on the first available record set
import numpy as np

if dataframes:
    example_id = list(dataframes.keys())[0]
    df = dataframes[example_id]

    # Try to select the first float/integer column (by heuristic)
    numeric_cols = [c for c in df.columns if np.issubdtype(df[c].dropna().astype(str).str.replace(',', '').astype(float, errors='ignore').dtype, np.number)]
    if not numeric_cols:
        # Try to auto-detect numeric-parseable columns (e.g., MW, Coverage, abundance, etc)
        for col in df.columns:
            # try to convert to numeric
            try:
                df_test = pd.to_numeric(df[col].dropna().astype(str).str.replace(',', ''), errors='raise')
                numeric_cols.append(col)
            except:
                continue
    print('Numeric field candidates (by @id):', numeric_cols)

    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field: {numeric_field}")

        # Clean and convert to float
        df[numeric_field] = pd.to_numeric(df[numeric_field].astype(str).str.replace(',', ''), errors='coerce')
        threshold = df[numeric_field].mean()  # Use mean as the threshold (or any domain-specific value)
        filtered_df = df[df[numeric_field] > threshold].copy()

        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by first non-numeric column if available
        non_numeric_cols = [c for c in df.columns if c not in numeric_cols]
        group_field = non_numeric_cols[0] if non_numeric_cols else None
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found to analyze.")
else:
    print("No data available for EDA. Check previous steps.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn where appropriate.

In [ ]:
# Visualization Example: histogram and scatter plot
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_cols:
    example_id = list(dataframes.keys())[0]
    df = dataframes[example_id]

    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot (if another numeric field exists)
    if len(numeric_cols) >= 2:
        plt.figure(figsize=(6, 4))
        sns.scatterplot(
            x=df[numeric_cols[0]], y=df[numeric_cols[1]])
        plt.title(f"{numeric_cols[0]} vs {numeric_cols[1]}")
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.show()
else:
    print("No numeric data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was loaded and explored using the `mlcroissant` library by referencing entities by their `@id`.
- Record sets and their field/column IDs were identified and used for DataFrame extraction and analysis.
- Basic EDA and visualizations allow insights into distributions and potential groupings within the data.
- This template can be extended with more record set-specific logic as needed.
